# 📊 Code 1c: Fetch Nifty Indices Data

## Purpose
Fetch **Close price** data for common Nifty indices from Yahoo Finance.

## Date Range
**Start Date:** 01-Jul-2009  
**End Date:** Today

## Output
- **`indices_raw_all.csv`** - All Nifty indices data
- `indices_not_found.csv` - Indices not available or with data quality issues

## Indices to Fetch:
- Nifty 50
- Nifty 100
- Nifty 200
- Nifty 500
- Nifty Midcap 50
- Nifty Midcap 100
- Nifty Midcap 150
- Nifty Smallcap 50
- Nifty Smallcap 100
- Nifty Smallcap 250
- Nifty Bank
- Nifty IT
- Nifty Pharma
- Nifty Auto
- Nifty FMCG
- Nifty Metal
- Nifty Energy
- Nifty Financial Services

---

**⏱️ Estimated Time:** 2-3 minutes

## Step 1: Install and Import Libraries

In [1]:
# Install required packages
!pip install yfinance --quiet
!pip install pandas --quiet

print("✅ Packages installed successfully!")

✅ Packages installed successfully!


In [2]:
import yfinance as yf
import pandas as pd
import numpy as np
from datetime import datetime
import warnings
warnings.filterwarnings('ignore')

print("✅ Libraries imported successfully!")
print(f"📅 Script run date: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")

✅ Libraries imported successfully!
📅 Script run date: 2026-06-15 18:21:38


## Step 2: Configuration

In [3]:
# ============================================================================
# CONFIGURATION
# ============================================================================

# Output files
OUTPUT_FILE = 'indices_raw_all.csv'
NOT_FOUND_FILE = 'indices_not_found.csv'

# Date range
START_DATE = '2007-01-01'
END_DATE = datetime.now().strftime('%Y-%m-%d')

# Nifty indices with their Yahoo Finance symbols
NIFTY_INDICES = {
    # Major Broad Indices
    'NIFTY_50': '^NSEI',
    'NIFTY_100': '^CNX100',
    'NIFTY_200': '^CNX200',
    'NIFTY_500': '^CRSLDX',

    # Midcap Indices
    'NIFTY_MIDCAP_50': 'NIFTY_MIDCAP_50.NS',
    'NIFTY_MIDCAP_100': '^NSEMDCP50',
    'NIFTY_MIDCAP_150': 'NIFTY_MIDCAP_150.NS',

    # Smallcap Indices
    'NIFTY_SMALLCAP_50': 'NIFTY_SMALLCAP_50.NS',
    'NIFTY_SMALLCAP_100': 'NIFTY_SMALLCAP_100.NS',
    'NIFTY_SMALLCAP_250': 'NIFTY_SMALLCAP_250.NS',

    # Sectoral Indices
    'NIFTY_BANK': '^NSEBANK',
    'NIFTY_IT': '^CNXIT',
    'NIFTY_PHARMA': '^CNXPHARMA',
    'NIFTY_AUTO': '^CNXAUTO',
    'NIFTY_FMCG': '^CNXFMCG',
    'NIFTY_METAL': '^CNXMETAL',
    'NIFTY_ENERGY': '^CNXENERGY',
    'NIFTY_FINANCIAL_SERVICES': 'NIFTY_FIN_SERVICE.NS',
    'NIFTY_REALTY': '^CNXREALTY',
    'NIFTY_MEDIA': '^CNXMEDIA',
}

print("="*80)
print("CODE 1c: FETCH NIFTY INDICES DATA")
print("="*80)
print(f"Output File: {OUTPUT_FILE}")
print(f"Date Range: {START_DATE} to {END_DATE}")
print(f"Total indices to fetch: {len(NIFTY_INDICES)}")
print("\nIndices:")
for name, symbol in NIFTY_INDICES.items():
    print(f"  - {name}: {symbol}")
print("="*80)

CODE 1c: FETCH NIFTY INDICES DATA
Output File: indices_raw_all.csv
Date Range: 2007-01-01 to 2026-06-15
Total indices to fetch: 20

Indices:
  - NIFTY_50: ^NSEI
  - NIFTY_100: ^CNX100
  - NIFTY_200: ^CNX200
  - NIFTY_500: ^CRSLDX
  - NIFTY_MIDCAP_50: NIFTY_MIDCAP_50.NS
  - NIFTY_MIDCAP_100: ^NSEMDCP50
  - NIFTY_MIDCAP_150: NIFTY_MIDCAP_150.NS
  - NIFTY_SMALLCAP_50: NIFTY_SMALLCAP_50.NS
  - NIFTY_SMALLCAP_100: NIFTY_SMALLCAP_100.NS
  - NIFTY_SMALLCAP_250: NIFTY_SMALLCAP_250.NS
  - NIFTY_BANK: ^NSEBANK
  - NIFTY_IT: ^CNXIT
  - NIFTY_PHARMA: ^CNXPHARMA
  - NIFTY_AUTO: ^CNXAUTO
  - NIFTY_FMCG: ^CNXFMCG
  - NIFTY_METAL: ^CNXMETAL
  - NIFTY_ENERGY: ^CNXENERGY
  - NIFTY_FINANCIAL_SERVICES: NIFTY_FIN_SERVICE.NS
  - NIFTY_REALTY: ^CNXREALTY
  - NIFTY_MEDIA: ^CNXMEDIA


## Step 3: Download Indices Data

Download Close price for all Nifty indices.

In [4]:
print("="*80)
print("DOWNLOADING NIFTY INDICES DATA")
print("="*80)
print(f"Date range: {START_DATE} to {END_DATE}")
print("\n⏳ Starting download...\n")

# Storage for data
all_indices_data = []
not_found_indices = []
data_quality_issues = []

# Download each index
for index_name, yahoo_symbol in NIFTY_INDICES.items():
    print(f"\nFetching {index_name} ({yahoo_symbol})...")

    try:
        # Download data
        data = yf.download(
            yahoo_symbol,
            start=START_DATE,
            end=END_DATE,
            auto_adjust=True,  # Get Close price only
            progress=False
        )

        if not data.empty and len(data) > 0:
            # Successfully downloaded
            data = data.reset_index()

            # Keep only Date and Close
            if 'Close' in data.columns:
                data = data[['Date', 'Close']].copy()
                data.columns = ['Date', 'Close']
            else:
                # Sometimes column name might be different
                data = data[['Date']].copy()
                data['Close'] = data.iloc[:, 1] if len(data.columns) > 1 else np.nan

            # Add index name
            data['Index'] = index_name

            # Check data quality
            missing_pct = data['Close'].isnull().sum() / len(data) * 100

            # Check date range
            first_date = data['Date'].min()
            last_date = data['Date'].max()

            # Store data
            all_indices_data.append(data)

            print(f"  ✅ Success: {len(data)} records")
            print(f"     Date range: {first_date.date()} to {last_date.date()}")
            print(f"     Missing data: {missing_pct:.2f}%")

            # Flag if data quality issues
            if first_date > pd.Timestamp(START_DATE) + pd.Timedelta(days=365):
                data_quality_issues.append({
                    'Index': index_name,
                    'Yahoo_Symbol': yahoo_symbol,
                    'Issue': f'Data starts from {first_date.date()} (later than {START_DATE})',
                    'First_Date': first_date,
                    'Last_Date': last_date,
                    'Total_Records': len(data)
                })
                print(f"     ⚠️  Warning: Data starts later than {START_DATE}")

            if missing_pct > 5:
                data_quality_issues.append({
                    'Index': index_name,
                    'Yahoo_Symbol': yahoo_symbol,
                    'Issue': f'High missing data: {missing_pct:.2f}%',
                    'First_Date': first_date,
                    'Last_Date': last_date,
                    'Total_Records': len(data)
                })
                print(f"     ⚠️  Warning: High missing data percentage")

        else:
            not_found_indices.append({
                'Index': index_name,
                'Yahoo_Symbol': yahoo_symbol,
                'Reason': 'No data available'
            })
            print(f"  ❌ No data available")

    except Exception as e:
        not_found_indices.append({
            'Index': index_name,
            'Yahoo_Symbol': yahoo_symbol,
            'Reason': str(e)[:100]
        })
        print(f"  ❌ Error: {str(e)[:100]}")

print("\n" + "="*80)
print("✅ DOWNLOAD COMPLETED!")
print("="*80)
print(f"Successfully downloaded: {len(all_indices_data)} indices")
print(f"Not found: {len(not_found_indices)} indices")
print(f"Data quality issues: {len(data_quality_issues)} indices")

DOWNLOADING NIFTY INDICES DATA
Date range: 2007-01-01 to 2026-06-15

⏳ Starting download...


Fetching NIFTY_50 (^NSEI)...
  ✅ Success: 4595 records
     Date range: 2007-09-17 to 2026-06-12
     Missing data: 0.00%

Fetching NIFTY_100 (^CNX100)...
  ✅ Success: 4780 records
     Date range: 2007-01-02 to 2026-06-12
     Missing data: 0.00%

Fetching NIFTY_200 (^CNX200)...
  ✅ Success: 4785 records
     Date range: 2007-01-02 to 2026-06-12
     Missing data: 0.00%

Fetching NIFTY_500 (^CRSLDX)...


ERROR:yfinance:
1 Failed download:
ERROR:yfinance:['NIFTY_MIDCAP_50.NS']: YFPricesMissingError('possibly delisted; no price data found  (1d 2007-01-01 -> 2026-06-15)')


  ✅ Success: 4780 records
     Date range: 2007-01-02 to 2026-06-12
     Missing data: 0.00%

Fetching NIFTY_MIDCAP_50 (NIFTY_MIDCAP_50.NS)...
  ❌ No data available

Fetching NIFTY_MIDCAP_100 (^NSEMDCP50)...


ERROR:yfinance:
1 Failed download:
ERROR:yfinance:['NIFTY_MIDCAP_150.NS']: YFPricesMissingError('possibly delisted; no price data found  (1d 2007-01-01 -> 2026-06-15)')


  ✅ Success: 4563 records
     Date range: 2007-09-24 to 2026-06-12
     Missing data: 0.00%

Fetching NIFTY_MIDCAP_150 (NIFTY_MIDCAP_150.NS)...
  ❌ No data available

Fetching NIFTY_SMALLCAP_50 (NIFTY_SMALLCAP_50.NS)...


...

  ❌ No data available

Fetching NIFTY_SMALLCAP_100 (NIFTY_SMALLCAP_100.NS)...


ERROR:yfinance:HTTP Error 404: {"quoteSummary":{"result":null,"error":{"code":"Not Found","description":"Quote not found for symbol: NIFTY_SMALLCAP_100.NS"}}}
ERROR:yfinance:
1 Failed download:
ERROR:yfinance:['NIFTY_SMALLCAP_100.NS']: YFTzMissingError('possibly delisted; no timezone found')


  ❌ No data available

Fetching NIFTY_SMALLCAP_250 (NIFTY_SMALLCAP_250.NS)...


ERROR:yfinance:
1 Failed download:
ERROR:yfinance:['NIFTY_SMALLCAP_250.NS']: YFTzMissingError('possibly delisted; no timezone found')


  ❌ No data available

Fetching NIFTY_BANK (^NSEBANK)...
  ✅ Success: 4610 records
     Date range: 2007-09-17 to 2026-06-12
     Missing data: 0.00%

Fetching NIFTY_IT (^CNXIT)...
  ✅ Success: 4610 records
     Date range: 2007-09-17 to 2026-06-12
     Missing data: 0.00%

Fetching NIFTY_PHARMA (^CNXPHARMA)...
  ✅ Success: 3782 records
     Date range: 2011-01-31 to 2026-06-12
     Missing data: 0.00%
     ⚠️  Warning: Data starts later than 2007-01-01

Fetching NIFTY_AUTO (^CNXAUTO)...
  ✅ Success: 3656 records
     Date range: 2011-07-12 to 2026-06-12
     Missing data: 0.00%
     ⚠️  Warning: Data starts later than 2007-01-01

Fetching NIFTY_FMCG (^CNXFMCG)...
  ✅ Success: 3767 records
     Date range: 2011-01-31 to 2026-06-12
     Missing data: 0.00%
     ⚠️  Warning: Data starts later than 2007-01-01

Fetching NIFTY_METAL (^CNXMETAL)...
  ✅ Success: 3656 records
     Date range: 2011-07-12 to 2026-06-12
     Missing data: 0.00%
     ⚠️  Warning: Data starts later than 2007-01-01


## Step 4: Combine Data into Wide Format

**Important:** Each index will be a separate column, with Date as the first column.

In [5]:
if len(all_indices_data) > 0:
    print("Combining all indices data into WIDE FORMAT...")
    print("-"*80)

    # Start with first index
    df_indices = all_indices_data[0][['Date', 'Close']].copy()
    first_index = all_indices_data[0]['Index'].iloc[0]
    df_indices = df_indices.rename(columns={'Close': first_index})

    # Merge remaining indices
    for i in range(1, len(all_indices_data)):
        temp_df = all_indices_data[i][['Date', 'Close']].copy()
        index_name = all_indices_data[i]['Index'].iloc[0]
        temp_df = temp_df.rename(columns={'Close': index_name})

        # Merge on Date
        df_indices = df_indices.merge(temp_df, on='Date', how='outer')

    # Sort by Date
    df_indices = df_indices.sort_values('Date').reset_index(drop=True)

    print(f"✅ Combined data in WIDE format!")
    print(f"   Total records (dates): {len(df_indices):,}")
    print(f"   Total columns: {len(df_indices.columns)} (1 Date + {len(df_indices.columns)-1} indices)")
    print(f"   Date range: {df_indices['Date'].min()} to {df_indices['Date'].max()}")
    print(f"\n   Columns: Date, {', '.join(df_indices.columns[1:])}")

    # Check missing values per index
    print(f"\n📊 MISSING VALUES PER INDEX:")
    for col in df_indices.columns[1:]:
        missing = df_indices[col].isnull().sum()
        missing_pct = missing / len(df_indices) * 100
        print(f"   {col}: {missing} missing ({missing_pct:.2f}%)")

else:
    print("❌ No data was downloaded successfully!")
    df_indices = pd.DataFrame()

Combining all indices data into WIDE FORMAT...
--------------------------------------------------------------------------------
✅ Combined data in WIDE format!
   Total records (dates): 4,787
   Total columns: 16 (1 Date + 15 indices)
   Date range: 2007-01-02 00:00:00 to 2026-06-12 00:00:00

   Columns: Date, NIFTY_50, NIFTY_100, NIFTY_200, NIFTY_500, NIFTY_MIDCAP_100, NIFTY_BANK, NIFTY_IT, NIFTY_PHARMA, NIFTY_AUTO, NIFTY_FMCG, NIFTY_METAL, NIFTY_ENERGY, NIFTY_FINANCIAL_SERVICES, NIFTY_REALTY, NIFTY_MEDIA

📊 MISSING VALUES PER INDEX:
   NIFTY_50: 192 missing (4.01%)
   NIFTY_100: 7 missing (0.15%)
   NIFTY_200: 2 missing (0.04%)
   NIFTY_500: 7 missing (0.15%)
   NIFTY_MIDCAP_100: 224 missing (4.68%)
   NIFTY_BANK: 177 missing (3.70%)
   NIFTY_IT: 177 missing (3.70%)
   NIFTY_PHARMA: 1005 missing (20.99%)
   NIFTY_AUTO: 1131 missing (23.63%)
   NIFTY_FMCG: 1020 missing (21.31%)
   NIFTY_METAL: 1131 missing (23.63%)
   NIFTY_ENERGY: 1019 missing (21.29%)
   NIFTY_FINANCIAL_SERVICES: 11

## Step 5: Handle Missing Values

Apply multiple strategies to handle missing values:
1. Forward fill for small gaps (1-3 days)
2. Linear interpolation for medium gaps (4-10 days)
3. Backward fill for any remaining gaps at the end

This ensures complete data for correlation analysis.

In [6]:
if not df_indices.empty:
    print("="*80)
    print("HANDLING MISSING VALUES")
    print("="*80)

    # Save raw version first (before preprocessing)
    print(f"\nSaving raw data to indices_raw_all.csv...")
    df_indices.to_csv('indices_raw_all.csv', index=False)
    print(f"✅ Raw data saved")

    # Count missing values before processing
    missing_before = df_indices.isnull().sum()
    total_missing_before = missing_before.sum()

    print(f"\n📊 BEFORE PROCESSING:")
    print(f"   Total missing values: {total_missing_before:,}")
    if total_missing_before > 0:
        print(f"\n   Missing values by index:")
        for col in df_indices.columns[1:]:
            if missing_before[col] > 0:
                pct = missing_before[col] / len(df_indices) * 100
                print(f"     {col}: {missing_before[col]} ({pct:.2f}%)")

    # Strategy 1: Forward fill for small gaps (1-3 days)
    print(f"\n🔄 Step 1: Forward fill (limit 3 days)...")
    for col in df_indices.columns[1:]:
        df_indices[col] = df_indices[col].fillna(method='ffill', limit=3)

    missing_after_ffill = df_indices.isnull().sum().sum()
    filled_ffill = total_missing_before - missing_after_ffill
    print(f"   Filled {filled_ffill} values")
    print(f"   Remaining: {missing_after_ffill}")

    # Strategy 2: Linear interpolation for medium gaps
    print(f"\n🔄 Step 2: Linear interpolation (limit 10 days)...")
    for col in df_indices.columns[1:]:
        df_indices[col] = df_indices[col].interpolate(
            method='linear',
            limit=10,
            limit_direction='both'
        )

    missing_after_interp = df_indices.isnull().sum().sum()
    filled_interp = missing_after_ffill - missing_after_interp
    print(f"   Filled {filled_interp} values")
    print(f"   Remaining: {missing_after_interp}")

    # Strategy 3: Backward fill for remaining gaps (especially at start/end)
    if missing_after_interp > 0:
        print(f"\n🔄 Step 3: Backward fill (no limit)...")
        for col in df_indices.columns[1:]:
            df_indices[col] = df_indices[col].fillna(method='bfill')

        missing_after_bfill = df_indices.isnull().sum().sum()
        filled_bfill = missing_after_interp - missing_after_bfill
        print(f"   Filled {filled_bfill} values")
        print(f"   Remaining: {missing_after_bfill}")

    # Final check
    missing_final = df_indices.isnull().sum()
    total_missing_final = missing_final.sum()

    print(f"\n✅ AFTER PROCESSING:")
    print(f"   Total missing values: {total_missing_final:,}")
    if total_missing_final > 0:
        print(f"\n   ⚠️  Remaining missing values:")
        for col in df_indices.columns[1:]:
            if missing_final[col] > 0:
                pct = missing_final[col] / len(df_indices) * 100
                print(f"     {col}: {missing_final[col]} ({pct:.2f}%)")
    else:
        print(f"   🎉 No missing values! Dataset is complete.")

    print(f"\n📊 SUMMARY:")
    print(f"   Total filled: {total_missing_before - total_missing_final:,}")
    print(f"   Fill rate: {(total_missing_before - total_missing_final) / total_missing_before * 100 if total_missing_before > 0 else 100:.2f}%")

    print("-"*80)
else:
    print("⚠️  No data to process")

HANDLING MISSING VALUES

Saving raw data to indices_raw_all.csv...
✅ Raw data saved

📊 BEFORE PROCESSING:
   Total missing values: 9,294

   Missing values by index:
     NIFTY_50: 192 (4.01%)
     NIFTY_100: 7 (0.15%)
     NIFTY_200: 2 (0.04%)
     NIFTY_500: 7 (0.15%)
     NIFTY_MIDCAP_100: 224 (4.68%)
     NIFTY_BANK: 177 (3.70%)
     NIFTY_IT: 177 (3.70%)
     NIFTY_PHARMA: 1005 (20.99%)
     NIFTY_AUTO: 1131 (23.63%)
     NIFTY_FMCG: 1020 (21.31%)
     NIFTY_METAL: 1131 (23.63%)
     NIFTY_ENERGY: 1019 (21.29%)
     NIFTY_FINANCIAL_SERVICES: 1170 (24.44%)
     NIFTY_REALTY: 884 (18.47%)
     NIFTY_MEDIA: 1148 (23.98%)

🔄 Step 1: Forward fill (limit 3 days)...
   Filled 193 values
   Remaining: 9101

🔄 Step 2: Linear interpolation (limit 10 days)...
   Filled 120 values
   Remaining: 8981

🔄 Step 3: Backward fill (no limit)...
   Filled 8981 values
   Remaining: 0

✅ AFTER PROCESSING:
   Total missing values: 0
   🎉 No missing values! Dataset is complete.

📊 SUMMARY:
   Total fille

## Step 6: Save Cleaned Output Files

Save the cleaned/preprocessed data to **indices_all.csv** (without 'raw' in name).

In [7]:
print("="*80)
print("SAVING CLEANED OUTPUT FILES")
print("="*80)

# Save cleaned indices data (preprocessed)
CLEANED_OUTPUT_FILE = 'indices_all.csv'

if not df_indices.empty:
    print(f"\nSaving cleaned data to {CLEANED_OUTPUT_FILE}...")
    df_indices.to_csv(CLEANED_OUTPUT_FILE, index=False)
    print(f"✅ Saved: {CLEANED_OUTPUT_FILE}")
    print(f"   Rows: {len(df_indices):,}")
    print(f"   Columns: {len(df_indices.columns)} (1 Date + {len(df_indices.columns)-1} indices)")
    print(f"   Missing values: {df_indices.isnull().sum().sum()}")

# Save not found / issues list
if len(not_found_indices) > 0 or len(data_quality_issues) > 0:
    # Combine not found and quality issues
    all_issues = not_found_indices + data_quality_issues
    df_issues = pd.DataFrame(all_issues)
    df_issues.to_csv(NOT_FOUND_FILE, index=False)
    print(f"\n📋 Saved: {NOT_FOUND_FILE}")
    print(f"   Indices with issues: {len(all_issues)}")

print("\n" + "="*80)
print("✅ ALL FILES SAVED!")
print("="*80)
print(f"\n📁 Output Files:")
print(f"   1. indices_raw_all.csv - Original data (before preprocessing)")
print(f"   2. {CLEANED_OUTPUT_FILE} - Cleaned data (after missing value handling)")
print(f"   3. {NOT_FOUND_FILE} - Issues report")

SAVING CLEANED OUTPUT FILES

Saving cleaned data to indices_all.csv...
✅ Saved: indices_all.csv
   Rows: 4,787
   Columns: 16 (1 Date + 15 indices)
   Missing values: 0

📋 Saved: indices_not_found.csv
   Indices with issues: 13

✅ ALL FILES SAVED!

📁 Output Files:
   1. indices_raw_all.csv - Original data (before preprocessing)
   2. indices_all.csv - Cleaned data (after missing value handling)
   3. indices_not_found.csv - Issues report


## Step 6.1: Verify Both Files Created

Check that both raw and cleaned files were created successfully.

In [8]:
import os

print("="*80)
print("FILE VERIFICATION")
print("="*80)

files_to_check = [
    ('indices_raw_all.csv', 'Raw data (before preprocessing)'),
    ('indices_all.csv', 'Cleaned data (after preprocessing)'),
    ('indices_not_found.csv', 'Issues report')
]

print("\nChecking output files:")
for filename, description in files_to_check:
    if os.path.exists(filename):
        size = os.path.getsize(filename) / 1024  # KB
        print(f"\n✅ {filename}")
        print(f"   {description}")
        print(f"   Size: {size:.2f} KB")

        # Load and show info
        try:
            df_check = pd.read_csv(filename)
            print(f"   Rows: {len(df_check):,}")
            print(f"   Columns: {len(df_check.columns)}")
            print(f"   Missing values: {df_check.isnull().sum().sum()}")
        except:
            pass
    else:
        print(f"\n❌ {filename} - NOT FOUND")
        print(f"   {description}")

print("\n" + "="*80)

FILE VERIFICATION

Checking output files:

✅ indices_raw_all.csv
   Raw data (before preprocessing)
   Size: 976.87 KB
   Rows: 4,787
   Columns: 16
   Missing values: 9294

✅ indices_all.csv
   Cleaned data (after preprocessing)
   Size: 1105.04 KB
   Rows: 4,787
   Columns: 16
   Missing values: 0

✅ indices_not_found.csv
   Issues report
   Size: 1.20 KB
   Rows: 13
   Columns: 7
   Missing values: 28



## Step 7: Display Summary

In [9]:
print("="*80)
print("FINAL SUMMARY - CODE 1c")
print("="*80)

print(f"""
📊 Download Statistics:
   - Total indices attempted: {len(NIFTY_INDICES)}
   - Successfully downloaded: {len(all_indices_data)}
   - Not found: {len(not_found_indices)}
   - Data quality issues: {len(data_quality_issues)}

📈 Output Datasets - WIDE FORMAT:

   1️⃣  indices_raw_all.csv (RAW - before preprocessing):
      - Total records (dates): {len(df_indices):,}
      - Total columns: {len(df_indices.columns) if not df_indices.empty else 0} (1 Date + {len(df_indices.columns)-1 if not df_indices.empty else 0} indices)
      - Date range: {df_indices['Date'].min() if not df_indices.empty else 'N/A'} to {df_indices['Date'].max() if not df_indices.empty else 'N/A'}

   2️⃣  indices_all.csv (CLEANED - after preprocessing):
      - Missing values handled using:
        * Forward fill (1-3 day gaps)
        * Linear interpolation (4-10 day gaps)
        * Backward fill (remaining gaps)
      - Final missing values: {df_indices.isnull().sum().sum() if not df_indices.empty else 0}
      - Format: Each index is a SEPARATE COLUMN
      - Ready for analysis and modeling

📁 Output Files:
   1. indices_raw_all.csv - Original data
   2. indices_all.csv - Cleaned/preprocessed data ⭐
   3. indices_not_found.csv - Issues and not found indices

➡️  Use indices_all.csv for benchmark comparisons and market analysis
""")

print("="*80)

FINAL SUMMARY - CODE 1c

📊 Download Statistics:
   - Total indices attempted: 20
   - Successfully downloaded: 15
   - Not found: 5
   - Data quality issues: 8

📈 Output Datasets - WIDE FORMAT:

   1️⃣  indices_raw_all.csv (RAW - before preprocessing):
      - Total records (dates): 4,787
      - Total columns: 16 (1 Date + 15 indices)
      - Date range: 2007-01-02 00:00:00 to 2026-06-12 00:00:00

   2️⃣  indices_all.csv (CLEANED - after preprocessing):
      - Missing values handled using:
        * Forward fill (1-3 day gaps)
        * Linear interpolation (4-10 day gaps)
        * Backward fill (remaining gaps)
      - Final missing values: 0
      - Format: Each index is a SEPARATE COLUMN
      - Ready for analysis and modeling

📁 Output Files:
   1. indices_raw_all.csv - Original data
   2. indices_all.csv - Cleaned/preprocessed data ⭐
   3. indices_not_found.csv - Issues and not found indices

➡️  Use indices_all.csv for benchmark comparisons and market analysis



## Step 7: Display Issues

In [10]:
if len(not_found_indices) > 0:
    print("\n📋 INDICES NOT FOUND:")
    print("="*80)
    df_not_found = pd.DataFrame(not_found_indices)
    display(df_not_found)

if len(data_quality_issues) > 0:
    print("\n⚠️  DATA QUALITY ISSUES:")
    print("="*80)
    df_quality = pd.DataFrame(data_quality_issues)
    display(df_quality)


📋 INDICES NOT FOUND:


,Index,Yahoo_Symbol,Reason
0,NIFTY_MIDCAP_50,NIFTY_MIDCAP_50.NS,No data available
1,NIFTY_MIDCAP_150,NIFTY_MIDCAP_150.NS,No data available
2,NIFTY_SMALLCAP_50,NIFTY_SMALLCAP_50.NS,No data available
3,NIFTY_SMALLCAP_100,NIFTY_SMALLCAP_100.NS,No data available
4,NIFTY_SMALLCAP_250,NIFTY_SMALLCAP_250.NS,No data available



⚠️  DATA QUALITY ISSUES:


,Index,Yahoo_Symbol,Issue,First_Date,Last_Date,Total_Records
0,NIFTY_PHARMA,^CNXPHARMA,Data starts from 2011-01-31 (later than 2007-0...,2011-01-31,2026-06-12,3782
1,NIFTY_AUTO,^CNXAUTO,Data starts from 2011-07-12 (later than 2007-0...,2011-07-12,2026-06-12,3656
2,NIFTY_FMCG,^CNXFMCG,Data starts from 2011-01-31 (later than 2007-0...,2011-01-31,2026-06-12,3767
3,NIFTY_METAL,^CNXMETAL,Data starts from 2011-07-12 (later than 2007-0...,2011-07-12,2026-06-12,3656
4,NIFTY_ENERGY,^CNXENERGY,Data starts from 2011-01-31 (later than 2007-0...,2011-01-31,2026-06-12,3768
5,NIFTY_FINANCIAL_SERVICES,NIFTY_FIN_SERVICE.NS,Data starts from 2011-09-07 (later than 2007-0...,2011-09-07,2026-06-12,3617
6,NIFTY_REALTY,^CNXREALTY,Data starts from 2010-07-19 (later than 2007-0...,2010-07-19,2026-06-12,3903
7,NIFTY_MEDIA,^CNXMEDIA,Data starts from 2011-08-04 (later than 2007-0...,2011-08-04,2026-06-12,3639


## Step 8: Preview Data

In [11]:
if not df_indices.empty:
    print("\nSAMPLE DATA (First 10 rows - WIDE FORMAT):")
    print("="*80)
    print("Note: Each index is a separate column")
    display(df_indices.head(10))

    print("\nLAST 10 ROWS (Recent data):")
    print("="*80)
    display(df_indices.tail(10))

    print("\nSUMMARY STATISTICS (All Indices):")
    print("="*80)
    display(df_indices.describe())


SAMPLE DATA (First 10 rows - WIDE FORMAT):
Note: Each index is a separate column


,Date,NIFTY_50,NIFTY_100,NIFTY_200,NIFTY_500,NIFTY_MIDCAP_100,NIFTY_BANK,NIFTY_IT,NIFTY_PHARMA,NIFTY_AUTO,NIFTY_FMCG,NIFTY_METAL,NIFTY_ENERGY,NIFTY_FINANCIAL_SERVICES,NIFTY_REALTY,NIFTY_MEDIA
0,2007-01-02,4494.649902,3878.100098,2101.219971,3323.095703,999.947449,6897.02002,4573.549805,4689.799805,3719.530029,8595.700195,3795.899902,8788.0,4010.679932,450.5,1346.680054
1,2007-01-03,4494.649902,3895.149902,2112.209961,3342.695557,999.947449,6897.02002,4573.549805,4689.799805,3719.530029,8595.700195,3795.899902,8788.0,4010.679932,450.5,1346.680054
2,2007-01-04,4494.649902,3864.100098,2101.209961,3327.895508,999.947449,6897.02002,4573.549805,4689.799805,3719.530029,8595.700195,3795.899902,8788.0,4010.679932,450.5,1346.680054
3,2007-01-05,4494.649902,3857.550049,2098.750000,3325.295654,999.947449,6897.02002,4573.549805,4689.799805,3719.530029,8595.700195,3795.899902,8788.0,4010.679932,450.5,1346.680054
4,2007-01-08,4494.649902,3808.850098,2072.409912,3286.895508,999.947449,6897.02002,4573.549805,4689.799805,3719.530029,8595.700195,3795.899902,8788.0,4010.679932,450.5,1346.680054
5,2007-01-09,4494.649902,3783.199951,2057.979980,3264.095703,999.947449,6897.02002,4573.549805,4689.799805,3719.530029,8595.700195,3795.899902,8788.0,4010.679932,450.5,1346.680054
6,2007-01-10,4494.649902,3729.500000,2026.839966,3214.595947,999.947449,6897.02002,4573.549805,4689.799805,3719.530029,8595.700195,3795.899902,8788.0,4010.679932,450.5,1346.680054
7,2007-01-11,4494.649902,3815.100098,2071.979980,3284.295654,999.947449,6897.02002,4573.549805,4689.799805,3719.530029,8595.700195,3795.899902,8788.0,4010.679932,450.5,1346.680054
8,2007-01-12,4494.649902,3919.449951,2122.899902,3362.095703,999.947449,6897.02002,4573.549805,4689.799805,3719.530029,8595.700195,3795.899902,8788.0,4010.679932,450.5,1346.680054
9,2007-01-15,4494.649902,3945.149902,2137.560059,3385.195557,999.947449,6897.02002,4573.549805,4689.799805,3719.530029,8595.700195,3795.899902,8788.0,4010.679932,450.5,1346.680054



LAST 10 ROWS (Recent data):


,Date,NIFTY_50,NIFTY_100,NIFTY_200,NIFTY_500,NIFTY_MIDCAP_100,NIFTY_BANK,NIFTY_IT,NIFTY_PHARMA,NIFTY_AUTO,NIFTY_FMCG,NIFTY_METAL,NIFTY_ENERGY,NIFTY_FINANCIAL_SERVICES,NIFTY_REALTY,NIFTY_MEDIA
4777,2026-06-01,23382.599609,24393.050781,13528.349609,22437.949219,17279.199219,53643.101562,29854.250000,24214.250000,25891.900391,48247.699219,13506.549805,40253.648438,25008.449219,768.200012,1427.550049
4778,2026-06-02,23483.550781,24491.300781,13577.000000,22521.099609,17287.300781,53714.648438,31116.550781,24006.300781,26079.449219,48612.800781,13557.650391,40189.648438,24861.250000,773.349976,1429.250000
4779,2026-06-03,23405.599609,24408.300781,13528.849609,22451.849609,17190.500000,54185.949219,29384.449219,24086.599609,26092.800781,48123.949219,13535.200195,40196.851562,24955.699219,762.599976,1420.800049
4780,2026-06-04,23416.550781,24427.699219,13549.849609,22497.699219,17264.650391,54307.851562,29300.599609,24177.949219,26144.349609,48216.000000,13436.000000,40445.750000,25031.250000,764.599976,1451.900024
4781,2026-06-05,23366.699219,24397.300781,13527.000000,22465.349609,17198.900391,54496.250000,29010.300781,24248.050781,26165.949219,48302.449219,13221.650391,40346.101562,25056.800781,768.900024,1502.449951
4782,2026-06-08,23123.000000,24111.099609,13362.250000,22173.699219,17039.150391,54063.750000,28653.550781,24147.650391,25681.800781,48098.648438,12913.650391,39685.449219,24805.000000,749.200012,1476.800049
4783,2026-06-09,23242.099609,24275.599609,13471.150391,22370.150391,17248.099609,55194.500000,28516.250000,24290.300781,26025.650391,48447.398438,12987.099609,39752.949219,25152.449219,761.400024,1474.599976
4784,2026-06-10,23214.949219,24200.750000,13397.799805,22233.849609,17005.199219,55100.300781,28279.949219,24160.750000,25833.550781,48957.000000,12766.450195,38949.300781,25206.250000,748.150024,1439.849976
4785,2026-06-11,23161.599609,24104.699219,13333.700195,22114.449219,16865.500000,55176.750000,27821.050781,24306.949219,25790.349609,48521.800781,12733.599609,38655.898438,25152.000000,743.349976,1465.500000
4786,2026-06-12,23622.900391,24602.250000,13618.549805,22599.800781,17265.900391,56814.800781,27795.750000,24380.050781,26293.849609,48827.601562,12854.500000,39244.449219,25943.349609,769.599976,1487.599976



SUMMARY STATISTICS (All Indices):


,Date,NIFTY_50,NIFTY_100,NIFTY_200,NIFTY_500,NIFTY_MIDCAP_100,NIFTY_BANK,NIFTY_IT,NIFTY_PHARMA,NIFTY_AUTO,NIFTY_FMCG,NIFTY_METAL,NIFTY_ENERGY,NIFTY_FINANCIAL_SERVICES,NIFTY_REALTY,NIFTY_MEDIA
count,4787,4787.000000,4787.000000,4787.000000,4787.000000,4787.000000,4787.000000,4787.000000,4787.000000,4787.000000,4787.000000,4787.000000,4787.000000,4787.000000,4787.000000,4787.000000
mean,2016-09-23 12:51:53.432212224,10969.155601,11081.700165,5849.318770,9437.149486,5556.939545,23841.036297,15963.511259,10210.926636,9491.123243,26144.860398,4313.924521,15694.378907,10731.826449,398.117025,1871.134548
min,2007-01-02 00:00:00,2524.199951,2388.600098,1258.069946,1966.847412,993.197754,3339.661133,2002.000000,4300.250000,3304.800049,8157.450195,1495.599976,6875.799805,3291.050049,128.250000,987.200012
25%,2011-11-14 12:00:00,5531.250000,5451.975098,2840.640015,4456.044189,2418.397827,10470.502930,6276.825195,4753.274902,3719.530029,10298.125000,2721.875000,8788.000000,4010.679932,227.075005,1346.680054
50%,2016-09-28 00:00:00,8629.150391,8754.200195,4535.049805,7224.190918,3645.008301,19043.779297,11173.400391,9291.349609,8159.000000,21658.949219,3795.899902,9915.500000,7845.750000,319.649994,1740.199951
75%,2021-08-05 12:00:00,15812.099609,16046.274902,8377.224609,13564.032227,7472.157227,35052.242188,27694.900391,12699.225098,11114.875000,36177.875000,5216.000000,20193.849609,16468.000000,450.500000,2235.000000
max,2026-06-12 00:00:00,26328.550781,27256.250000,14789.349609,24496.900391,17754.949219,61550.800781,45995.800781,24890.900391,28922.349609,66305.203125,13718.299805,44954.449219,28463.250000,1150.300049,3642.699951
std,NaN,6582.994363,6826.039347,3710.449752,6186.561213,4521.724204,15495.376327,11812.655403,5387.308073,6551.947319,16020.885343,2424.933427,9977.048641,7212.074802,231.775595,569.493642


## 📥 Download Files (Optional)

In [12]:
from google.colab import files

print("Downloading files...")
files.download(OUTPUT_FILE)
if len(not_found_indices) > 0 or len(data_quality_issues) > 0:
    files.download(NOT_FOUND_FILE)

files.download("indices_all.csv")

print("\n✅ Download complete!")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>


✅ Download complete!


---

## ✅ Code 1c Complete!

**Next Step:** Run **Code 1d** to fetch commodities data.